# TrOCR-style model on Google Colab (Prabhat)

Before you start:
1. **Runtime menu -> Change runtime type -> Hardware accelerator = T4 GPU -> Save.**
2. **Left sidebar -> the key icon (Secrets) -> Add new secret:** name `GITHUB_TOKEN`, value = the token Nikhil gave you, and switch **Notebook access** on.
3. Run the cells **one at a time, in order**. **Do not use "Run all"**: cell 8 is the 3-hour full run.
4. Keep this browser tab open while a cell is running (Colab disconnects idle or closed tabs). Checkpoints are saved to your Google Drive, so after a disconnect you only run cells 1, 2, 4 (mount Drive) and the same cell 8 with `--resume`.

Harmless messages: the Hugging Face *unauthenticated requests* warning, `encoder.pooler ... MISSING`, and a `lr_scheduler.step()` warning in the pytest output.

In [ ]:
# 1. GPU check: must print True and a GPU name
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# 2. get the code (private repo) and install the few missing packages. SAFE TO RE-RUN: never clones twice.
import os
from google.colab import userdata
REPO = "/content/Devanagari_Word_Recognition_Project"
if not os.path.exists(REPO):
    token = userdata.get("GITHUB_TOKEN")
    !git clone https://{token}@github.com/Nikhil04-ai/Devanagari_Word_Recognition_Project.git {REPO}
%cd {REPO}
!git remote set-url origin https://github.com/Nikhil04-ai/Devanagari_Word_Recognition_Project.git   # remove the token from the saved remote
!pip install -q rapidfuzz pyyaml accelerate pytest

In [ ]:
# 3. data: IIIT-HW-Dev Version 1 (about 1.8 GB, two archives), 8 parallel connections. Wait for DONE.
%cd /content/Devanagari_Word_Recognition_Project
!python scripts/download_iiit.py --out /content/IIIT-HW-Hindi_v1.tar.gz
!mkdir -p data/raw/iiit_hw_dev
!tar -xzf /content/IIIT-HW-Hindi_v1.tar.gz -C data/raw/iiit_hw_dev
!tar -xzf data/raw/iiit_hw_dev/HindiSeg.tar.gz -C data/raw/iiit_hw_dev
!rm /content/IIIT-HW-Hindi_v1.tar.gz data/raw/iiit_hw_dev/HindiSeg.tar.gz

In [ ]:
# 4. mount Google Drive (checkpoints are kept there, so a disconnect does not lose the run)
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/p09_trocr

In [ ]:
# 5. check the data (expect 69853 / 12708 / 12869 and no missing images), then unit tests
%cd /content/Devanagari_Word_Recognition_Project
import csv, os
bad = 0
for split in ("train", "val", "test"):
    rows = list(csv.DictReader(open(f"data/processed/words/{split}.csv", encoding="utf-8")))
    missing = sum(not os.path.exists(r["image_path"]) for r in rows[:2000])
    bad += missing
    print(split, len(rows), "rows | missing among the first 2000:", missing)
assert bad == 0, "images not found: run cell 3 in this folder, then: !ls data/raw/iiit_hw_dev/HindiSeg | head"
print("data OK")
!python -m pytest -q

In [ ]:
# 6. SANITY RUN: the model must memorise 100 training images (CER -> 0, it stops by itself). Takes a few minutes.
# If it does not reach CER 0: check pad=0 / bos=1 / eos=2 ids, that padded labels are -100, and the learning rates.
%cd /content/Devanagari_Word_Recognition_Project
!python -m src.trocr.train --config configs/trocr/overfit.yaml

In [ ]:
# 7. (optional) first look at speed: nothing to run. Open the log of cell 8 after the FIRST epoch and note the minutes.
# Too slow (more than about 40 min per epoch)? Use the subset command written in cell 8.

In [ ]:
# 8. FULL RUN. Start only after cell 6 printed "overfit test passed". Time budget: max_minutes = 190 (config).
# Checkpoints go to Google Drive. To resume after a disconnect: run cells 1, 2, 4, then this same cell with --resume added.
%cd /content/Devanagari_Word_Recognition_Project
!python -m src.trocr.train --config configs/trocr/full.yaml --ckpt_dir /content/drive/MyDrive/p09_trocr/trocr_full
# if one epoch is too slow, stop and use this instead (report the subset honestly):
# !python -m src.trocr.train --config configs/trocr/full.yaml --ckpt_dir /content/drive/MyDrive/p09_trocr/trocr_full --train_limit 30000 --epochs 4
# resume (epochs / train_limit must stay the same as the original run):
# !python -m src.trocr.train --config configs/trocr/full.yaml --ckpt_dir /content/drive/MyDrive/p09_trocr/trocr_full --resume

In [ ]:
# 9. evaluate on the TEST set ONCE; writes results/trocr_full_greedy/
%cd /content/Devanagari_Word_Recognition_Project
!python -m src.trocr.evaluate --ckpt /content/drive/MyDrive/p09_trocr/trocr_full/best.pt

In [ ]:
# 10. numbers and learning curve for report/sections/06_trocr.md (take values from here, do not invent any)
%cd /content/Devanagari_Word_Recognition_Project
import json, pandas as pd
print(json.dumps(json.load(open("results/trocr_full_greedy/metrics.json", encoding="utf-8")), indent=1))
log = pd.read_csv("/content/drive/MyDrive/p09_trocr/trocr_full/log.csv")
print(log)
ax = log.plot(x="epoch", y=["train_loss", "val_loss"], title="TrOCR loss"); ax.figure.savefig("trocr_loss.png", dpi=150, bbox_inches="tight")
ax = log.plot(x="epoch", y="val_cer", title="TrOCR validation CER"); ax.figure.savefig("trocr_val_cer.png", dpi=150, bbox_inches="tight")

In [ ]:
# 11. keep the results in Google Drive, then download results/trocr_full_greedy/ (CSV + json) and the two PNGs.
# Commit only results/trocr_full_greedy/ (small); never commit best.pt / last.pt.
%cd /content/Devanagari_Word_Recognition_Project
!cp -r results/trocr_full_greedy trocr_loss.png trocr_val_cer.png /content/drive/MyDrive/p09_trocr/
!ls -la /content/drive/MyDrive/p09_trocr